In [1]:
import os
import gc
import pandas as pd
from obspy import read, UTCDateTime

# ==============================================================================
# 🎛️ PARAMETER JALUR ABSOLUT BERKAS DI SSD MAC BAPAK
# ==============================================================================
PATH_QUEUE_READY = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/FINAL_DOWNLOAD_QUEUE_READY.csv'
INPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'
OUTPUT_STEAD_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/sliced_stead_dataset'

# Konfigurasi Dimensi Sinyal Latih AI
TARGET_SAMPLING_RATE = 100.0  # Resample ke 100 Hz standar STEAD
DURATION_SEC = 60.0           # Durasi potongan 60 detik (Tepat 6000 sampel)

def jalankan_slicing_dan_simpan_mseed():
    print("="*90)
    print("🚀 STARTING: SEISMIC WAVEFORM SLICER PIPELINE (WRITE TO LOCAL MSEED)")
    print("="*90)
    
    if not os.path.exists(PATH_QUEUE_READY):
        print(f"❌ GALAT: Berkas antrean '{PATH_QUEUE_READY}' tidak ditemukan!")
        return
        
    print("⏳ Memuat panduan antrean berkas tugas berstasiun sejati...")
    df_queue = pd.read_csv(PATH_QUEUE_READY)
    
    # Menyiapkan folder lokal terpisah berdasarkan kategori standar STEAD
    dir_le = os.path.join(OUTPUT_STEAD_DIR, 'LE')
    dir_no = os.path.join(OUTPUT_STEAD_DIR, 'NO')
    os.makedirs(dir_le, exist_ok=True)
    os.makedirs(dir_no, exist_ok=True)
    
    total_proses, sukses_le, sukses_no = 0, 0, 0
    expected_samples = int(DURATION_SEC * TARGET_SAMPLING_RATE)  # Wajib 6000 titik data
    
    print("⏳ Menjalankan pemotongan fasa getaran seismik lokal secara otonom...")
    
    for row in df_queue.itertuples():
        eid = row.Event_ID
        net = row.Net
        sta = row.Station
        time_str = row.Time_UTC
        
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        
        # Susun alamat lokasi file biner mentah di SSD Mac Anda
        file_mseed_path = os.path.join(INPUT_WAVEFORM_DIR, year_folder, str(eid), f"{net}_{sta}_{eid}.mseed")
        
        if not os.path.exists(file_mseed_path) or os.path.getsize(file_mseed_path) == 0:
            continue
            
        try:
            total_proses += 1
            st = read(file_mseed_path)
            
            # --- TAHAP 1: PRAPEMROSESAN DIGITAL SINYAL ---
            st.detrend("demean")
            st.detrend("linear")
            st.resample(TARGET_SAMPLING_RATE)
            
            # --- TAHAP 2: EKSTRAKSI JENDELA NO (NOISE) STANDAR STEAD ---
            # Aturan: t_event - 60 detik s.d t_event
            start_no = t_event - DURATION_SEC
            end_no = t_event
            st_no = st.slice(starttime=start_no, endtime=end_no)
            
            # --- TAHAP 3: EKSTRAKSI JENDELA LE (LOCAL EARTHQUAKE) STANDAR STEAD ---
            # Aturan: t_event s.d t_event + 60 detik
            start_le = t_event
            end_le = t_event + DURATION_SEC
            st_le = st.slice(starttime=start_le, endtime=end_le)
            
            # --- TAHAP 4: FILTER INTEGRITAS DIMENSI & PENULISAN BERKAS LOCAL ---
            # Ekspor fasa gempa lokal (LE) kembali ke .mseed terpisah
            if len(st_le) == 3 and all(len(tr.data) >= expected_samples for tr in st_le):
                out_name_le = f"{net}_{sta}_{eid}_LE.mseed"
                st_le.write(os.path.join(dir_le, out_name_le), format="MSEED")
                sukses_le += 1
                
            # Ekspor fasa bising lingkungan (NO) kembali ke .mseed terpisah
            if len(st_no) == 3 and all(len(tr.data) >= expected_samples for tr in st_no):
                out_name_no = f"{net}_{sta}_{eid}_NO.mseed"
                st_no.write(os.path.join(dir_no, out_name_no), format="MSEED")
                sukses_no += 1
                
            del st, st_le, st_no
            
        except Exception:
            continue
            
        # Kosongkan cache memori laptop setiap 500 file terproses agar sistem tetap enteng
        if total_proses % 500 == 0:
            gc.collect()
            print(f"   • Progres: {total_proses:,} file biner mseed berhasil dibedah...")
            
    print("\n📊 LAPORAN AKHIR PEMOTONGAN DATASET STANDAR STEAD LOKAL:")
    print("-" * 75)
    print(f"   • Total File Mentah Teridentifikasi : {total_proses:,} file .mseed")
    print(f"   • Sukses Terbit File Baru Label LE  : {sukses_le:,} berkas biner .mseed")
    print(f"   • Sukses Terbit File Baru Label NO  : {sukses_no:,} berkas biner .mseed")
    print("-" * 75)
    print(f"✅ PIPELINE PARIPURNA: File biner tersimpan rapi di direktori -> {OUTPUT_STEAD_DIR}")
    print("="*90)
    
    gc.collect()

if __name__ == "__main__":
    jalankan_slicing_dan_simpan_mseed()

🚀 STARTING: SEISMIC WAVEFORM SLICER PIPELINE (WRITE TO LOCAL MSEED)
⏳ Memuat panduan antrean berkas tugas berstasiun sejati...
⏳ Menjalankan pemotongan fasa getaran seismik lokal secara otonom...


/opt/homebrew/Caskroom/miniforge/base/envs/waveform/lib/python3.10/site-packages/obspy/io/mseed/core.py:1034: UserWarning: The encoding specified in trace.stats.mseed.encoding does not match the dtype of the data.
A suitable encoding will be chosen.
  warnings.warn(msg, UserWarning)


   • Progres: 500 file biner mseed berhasil dibedah...

📊 LAPORAN AKHIR PEMOTONGAN DATASET STANDAR STEAD LOKAL:
---------------------------------------------------------------------------
   • Total File Mentah Teridentifikasi : 985 file .mseed
   • Sukses Terbit File Baru Label LE  : 557 berkas biner .mseed
   • Sukses Terbit File Baru Label NO  : 555 berkas biner .mseed
---------------------------------------------------------------------------
✅ PIPELINE PARIPURNA: File biner tersimpan rapi di direktori -> /Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/sliced_stead_dataset
